# GeoZarr v3 parameter discovery (chunk / shard / codec)

The **only** legitimate notebook in this harness (per the D3 scaffold plan): a
thin scaffold to settle the Zarr v3 sharding levers on real MAJA L2A granularity
*before* they are codified into the adapter defaults and the testbed configs. It
is exploratory and not part of CI; the benchmark itself runs through the harness
(`cng-benchmark`), never here.

What we settle here:

- **chunk shape** — the addressable (range-read) unit. Small enough that a map
  tile pulls few chunks, large enough to keep per-chunk overhead down.
- **shard shape** — the stored object (a shard packs many chunks). This is the
  lever that lifts the mean object size over the Datalake tiers (Tier 2 ≥ 32 MB,
  Tier 3 ≥ 100 MB).
- **codec** — the per-chunk compressor (zstd by default).

Run against a Scaleway bucket copy of a MAJA granule, not the VRE.

In [ ]:
# Requires the geozarr + cog extras: uv sync --extra geozarr --extra cog
import numpy as np
import rioxarray  # noqa: F401

from cng_benchmark.formats.geozarr import (
    GeoZarrAdapter,
    describe_store_layout,
    enumerate_store_objects,
)

# Point this at one real MAJA reflectance band read on the fly, e.g.
#   /vsizip//vsis3/sentinel2-l2a-sprid/T31TCJ/<scene>.zip/<...>_FRE_B4.tif
SOURCE = "/vsizip//vsis3/<bucket>/<scene>.zip/<...>_FRE_B4.tif"

In [ ]:
# Sweep candidate chunk/shard shapes and codecs; report the resulting object-size
# distribution and shard count against the tier thresholds.
TIER2, TIER3 = 32 * 2**20, 100 * 2**20
candidates = [
    {"chunk_shape": [1024, 1024], "shard_shape": [1024, 1024], "codec": "zstd"},
    {"chunk_shape": [1024, 1024], "shard_shape": [2048, 2048], "codec": "zstd"},
    {"chunk_shape": [512, 512], "shard_shape": [2048, 2048], "codec": "zstd"},
    {"chunk_shape": [1024, 1024], "shard_shape": [2048, 2048], "codec": "none"},
]

for params in candidates:
    target = f"/tmp/discovery-{params['shard_shape'][0]}-{params['codec']}.zarr"
    GeoZarrAdapter().convert(SOURCE, target, params)
    sizes = np.array(enumerate_store_objects(target))
    ly = describe_store_layout(target, "FRE_B4")
    print(
        params,
        f"shards={ly.shard_count}",
        f"mean={sizes.mean() / 2**20:.1f} MiB",
        f"≥T2={(sizes >= TIER2).mean():.0%}",
        f"≥T3={(sizes >= TIER3).mean():.0%}",
    )

Pick the smallest shard that keeps the mean shard size in the target tier, then
codify it as the defaults in `configs/datasets/*.yaml` /
`configs/benchmarks/*geozarr.yaml` and the testbed arm. From here on the numbers
are produced by the harness, not this notebook.